# FD001 Initial EDA
This notebook is Task 2 for the 10-day checkpoint. It inspects only the raw NASA C-MAPSS FD001 files and does not make model-selection decisions from official test labels.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from predictive_maintenance.data import read_cmapss_file
from predictive_maintenance.constants import SENSOR_COLUMNS
train = read_cmapss_file(ROOT / 'data/raw/train_FD001.txt')
test = read_cmapss_file(ROOT / 'data/raw/test_FD001.txt')
train.shape, test.shape


## Basic structure


In [ ]:
display(train.head())
print('Training engines:', train.engine_id.nunique())
print('Test engines:', test.engine_id.nunique())
print('Missing train cells:', int(train.isna().sum().sum()))
print('Missing test cells:', int(test.isna().sum().sum()))
display(train.describe().T)


## Engine trajectory lengths


In [ ]:
lengths = train.groupby('engine_id')['cycle'].max()
display(lengths.describe())
lengths.hist(bins=20)
plt.title('FD001 training trajectory lengths')
plt.xlabel('Final cycle')
plt.ylabel('Engine count')
plt.show()


## Sensor variability
This is descriptive only. The frozen removal list is produced later using the **80 training engines only**, after the development split is fixed.


In [ ]:
sensor_summary = pd.DataFrame({
    'nunique': train[SENSOR_COLUMNS].nunique(),
    'mean': train[SENSOR_COLUMNS].mean(),
    'std': train[SENSOR_COLUMNS].std(),
    'min': train[SENSOR_COLUMNS].min(),
    'max': train[SENSOR_COLUMNS].max(),
}).sort_values('std')
display(sensor_summary)


## Example sensor histories


In [ ]:
example_engine = int(train.engine_id.iloc[0])
example = train[train.engine_id == example_engine]
for sensor in SENSOR_COLUMNS[:6]:
    plt.figure()
    plt.plot(example['cycle'], example[sensor])
    plt.title(f'Engine {example_engine}: {sensor}')
    plt.xlabel('Cycle')
    plt.ylabel(sensor)
    plt.show()


## EDA findings to record
After execution, summarize: dataset dimensions, engine counts, trajectory-length range, whether raw files contain missing values, visibly constant/low-variance sensors, and any unusual distributions. Do **not** use official test RUL values to decide preprocessing or model configuration.
